In [ ]:
import pandas as pd
import numpy
import xlrd


# Framing

**Business inquiry**: What is the probability that a client will default on their next payment?
    + (This is key intel on how the decisions on a client's credit limit are defined)

Thus **target variable** is `default payment next month` (renamed to `default`)
    p(defaulting) $\in$ (0,1)
    p() used by costing, pricing, E(loss)

The data is a flat spreadsheet with unit client/row.

**cost ratio** is a stated assumption, e.g. 5:1
    - config

# Data Checks

## Import data

In [ ]:
df = pd.read_excel("../data/raw/uci_taiwan_default.xls")
df.head()

## Check for missing values or duplicates

In [ ]:
print('Entries:', len(df))
print('Total missing:', df.isna().sum().sum())
# print('Duplicated rows:', df.duplicated().tolist())

for i in df.duplicated().tolist():
    dupes = 0
    if i != False:
        dupes += 1
print('Duplicated entries:', dupes)

There are no duplicates or missing values.

The only fix would be dropping the top aliases and changing the name of the target to 'defaults'.

This will be managed by `data.py`

In [ ]:
dupes = 0
for i in df.drop(columns="Unnamed: 0").duplicated().tolist():
    if i != False:
        dupes += 1
print('Duplicated entries (ignoring ID):', dupes)


35 registers look the same, but this is not enough to matter much. Some might be inactive accounts. They are real people, so it is representative of what banking data would be like.

## Exploratory analysis

In [ ]:
from credit_risk.data import load_raw, TARGET
df = load_raw()

In [ ]:
from credit_risk.data import load_raw, clean, make_split, load_splits, TARGET

df = clean(load_raw())
make_split(df) # run once; afterwards only load_splits
# split ratios and seed in configs/uci_taiwan.toml
splits = load_splits(df)
train, validation = splits["train"], splits["validation"]


In [ ]:
# inspect
print(train)
print(validation)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from credit_risk.config import PROJECT_ROOT
from credit_risk.data import TARGET, clean, load_raw, load_splits

plt.style.use("ggplot")
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = clean(load_raw())
splits = load_splits(df)
train = splits["train"]

LABELS = {
    "SEX": {1: "Male", 2: "Female"},
    "EDUCATION": {1: "Graduate school", 2: "University", 3: "High school", 4: "Other"},
    "MARRIAGE": {1: "Married", 2: "Single", 3: "Other"},
}
PAY_COLS = [f"PAY_{i}" for i in range(1, 7)]
BASE_RATE = train[TARGET].mean()


In [ ]:
balance = train[TARGET].value_counts().to_frame("clients")
balance["share"] = balance["clients"] / balance["clients"].sum()
balance.style.format({"share": "{:.1%}"})


From this we get:
- 14,018 non-defaulting clients vs 3982 defaulting ones.
- 77.9% non-default
- 22.1% default


In [ ]:
# Descriptive statistics!
train.drop(columns=TARGET).describe().T.style.format("{:,.1f}")


In [ ]:
def default_rate_table(data: pd.DataFrame, col: str) -> pd.DataFrame:
    table = data.groupby(col)[TARGET].agg(clients="size", defaults="sum", rate="mean")
    table["share"] = table["clients"] / table["clients"].sum()
    return table.rename(index=LABELS.get(col, {}))


for col in ["SEX", "EDUCATION", "MARRIAGE", "PAY_1"]:
    print(col)
    print(default_rate_table(train, col).round(3), "\n")



In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
for ax, col in zip(axes.flat, PAY_COLS):
    table = default_rate_table(train, col)
    small = table["clients"] < 50
    ax.bar(table.index, table["rate"], color=np.where(small, "#BBBBBB", "#348ABD"))
    ax.axhline(BASE_RATE, color="#555555", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.set_xticks(table.index)
axes[0, 0].set_ylabel("Default rate")
axes[1, 0].set_ylabel("Default rate")
for ax in axes[1]:
    ax.set_xlabel("Repayment status")
fig.suptitle(
    "Default rate by repayment status (grey: fewer than 50 clients; dashed: overall rate)"
)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_default_rate_by_pay_status.png", dpi=150, bbox_inches="tight")


In [ ]:
age_band = pd.cut(train["AGE"], bins=[20, 25, 30, 35, 40, 50, 60, 80])
demo = train.assign(AGE_BAND=age_band)

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, ["SEX", "EDUCATION", "MARRIAGE", "AGE_BAND"]):
    table = default_rate_table(demo, col)
    ax.bar(table.index.astype(str), table["rate"], color="#348ABD")
    ax.axhline(BASE_RATE, color="#555555", linestyle="--", linewidth=1)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Default rate")
fig.suptitle("Default rate by demographic group (dashed: overall rate)")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_default_rate_by_demographics.png", dpi=150, bbox_inches="tight")


In [ ]:
def signed_log10(x: pd.Series) -> pd.Series:
    return np.sign(x) * np.log10(1 + x.abs())


panels = [
    ("LIMIT_BAL", np.log10(train["LIMIT_BAL"]), "log10(LIMIT_BAL)", 40),
    ("AGE", train["AGE"], "AGE (years)", np.arange(20, 82, 2)),
    ("BILL_AMT1", signed_log10(train["BILL_AMT1"]), "signed log10(1 + |BILL_AMT1|)", 60),
    ("PAY_AMT1", signed_log10(train["PAY_AMT1"]), "log10(1 + PAY_AMT1)", 60),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, values, xlabel, bins) in zip(axes.flat, panels):
    for outcome, colour, name in [(0, "#348ABD", "No default"), (1, "#E24A33", "Default")]:
        ax.hist(
            values[train[TARGET] == outcome],
            bins=bins,
            density=True,
            histtype="step",
            linewidth=2,
            color=colour,
            label=name,
        )
    ax.set_title(col)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
axes[0, 0].legend()
fig.suptitle("Distributions by outcome")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_distributions_by_outcome.png", dpi=150, bbox_inches="tight")



### Correlation Matrix
Spearman is correlation computed on ranks, thus skewed ratios won't dominate.

In [ ]:
corr = train.corr(method="spearman")

fig, ax = plt.subplots(figsize=(13, 11))
image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr)), corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        value = corr.iloc[i, j]
        ax.text(
            j, i, f"{value:.2f}", ha="center", va="center", fontsize=6,
            color="white" if abs(value) > 0.6 else "black",
        )
fig.colorbar(image, ax=ax, shrink=0.8, label="Spearman correlation")
ax.set_title("Spearman correlation, training set")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_correlation_matrix.png", dpi=150, bbox_inches="tight")


Bills trivially correlate to each other, which is to be disregarded. Asides from that, they don't correlate to anything else very strongly. The target correlates mostly with payment status (PAY_1 to PAY_6). 

In [ ]:
(
    corr[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
    .to_frame("spearman")
    .style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1)
    .format("{:.3f}")
)


Men default a bit more, and age shows (potentially) higher risk at the extremes: very young (early 20s) or older than 40 people seem to default more. It is a protected characteristic, I'll drop it for now.

Sex, Marriage and Age are as well, and add very little. They'll be dropped for the models, but kept for the fairness analysis.

Men have a default rate of 24.3%, women 20.7%. 

Education has the rates as follows: 19.2% graduate school, university 23.5%, high school 25.6%, other 9.7% (on mere 279 clients)

### **pay-status:**

13% default at status 0 vs 70% at 2 months late

This looks like the most likely predictor.

# Add ratios as new features?

In this case I might try some feature engineering.

In [ ]:
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from optbinning import BinningProcess
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

from credit_risk.data import TARGET, clean, load_raw, load_splits

warnings.filterwarnings("ignore", category=UserWarning)

PROTECTED = ["SEX", "MARRIAGE", "AGE"]


*utilisation ratio* : billing amount / limit credit
    If it is too high it means the account is being maxed out, which is bad. It also corrects for what a big amount means to a richer or poorer client

*repay ratio* : paid amount this month / billed amount last month
    I.E. share of the last month statement that was paid off the next

(left blank when the previous bill is 0) 

In [ ]:
df = clean(load_raw()).drop(columns=PROTECTED)

for i in range(1, 7):
    df[f"UTIL_{i}"] = df[f"BILL_AMT{i}"] / df["LIMIT_BAL"]
for i in range(1, 6):
    previous_bill = df[f"BILL_AMT{i + 1}"]
    df[f"REPAY_RATIO_{i}"] = (df[f"PAY_AMT{i}"] / previous_bill).where(previous_bill > 0)

splits = load_splits(df)
train, validation = splits["train"], splits["validation"]

ratio_cols = [c for c in df.columns if c.startswith(("UTIL_", "REPAY_RATIO_"))]
raw_cols = [c for c in df.columns if c != TARGET and c not in ratio_cols]
feature_sets = {"raw only": raw_cols, "raw + ratios": raw_cols + ratio_cols}


In [ ]:
def fit_predict_scorecard(cols: list[str]) -> np.ndarray:
    binning = BinningProcess(
        cols,
        categorical_variables=["EDUCATION"],
        selection_criteria={"iv": {"min": 0.02}},
    )
    binning.fit(train[cols], train[TARGET])
    model = LogisticRegression(max_iter=1000)
    model.fit(binning.transform(train[cols]), train[TARGET])
    return model.predict_proba(binning.transform(validation[cols]))[:, 1]


def fit_predict_lightgbm(cols: list[str]) -> np.ndarray:
    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=50,
        random_state=0,
        verbose=-1,
    )
    model.fit(train[cols], train[TARGET])
    return model.predict_proba(validation[cols])[:, 1]


def bootstrap_auc_gain(y, p_base, p_new, n=500, seed=0):
    rng = np.random.default_rng(seed)
    gains = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        gains.append(roc_auc_score(y[idx], p_new[idx]) - roc_auc_score(y[idx], p_base[idx]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return low, high


In [ ]:
y_val = validation[TARGET].to_numpy()
models = {"WoE scorecard": fit_predict_scorecard, "LightGBM": fit_predict_lightgbm}

rows = []
for model_name, fit_predict in models.items():
    predictions = {name: fit_predict(cols) for name, cols in feature_sets.items()}
    low, high = bootstrap_auc_gain(y_val, predictions["raw only"], predictions["raw + ratios"])
    for name, p in predictions.items():
        auc = roc_auc_score(y_val, p)
        rows.append(
            {
                "model": model_name,
                "features": name,
                "AUC": auc,
                "Gini": 2 * auc - 1,
                "Brier": brier_score_loss(y_val, p),
                "AUC gain 95% CI": f"[{low:+.4f}, {high:+.4f}]" if name != "raw only" else "",
            }
        )

results = pd.DataFrame(rows).set_index(["model", "features"])
results.style.format({"AUC": "{:.4f}", "Gini": "{:.4f}", "Brier": "{:.4f}"})


### Table interpretation

**AUC (Area under the curve)**
Probability that the model gave the defaulter the higher risk score.
    
$\iff$ 0.5 $\Rightarrow$ the model is no better than a coin toss.

$\iff$ 0.5 $\Rightarrow$ ranks every defaulter above non-defaulters.

**Gini**
$2*\text{AUC}-1$
0 $\Rightarrow$ coin toss
1 $\Rightarrow$ perfect

0.52 is apparently a decent value here, moreso for an untuned model.

**Brier score**

$\forall$ clients := $(p[\text{default}]-\text{real outcome})^2$ 

Where outcome=1 := defaulting ,$\land$, outcome := 0 not defaulting

Unlike AUC or Gini, it penalises probabilities with the wrong size. Calibration and Ranking are considered:

p=0.8 , defaults $\Rightarrow$ Brier = 0.04
not defaul $\Rightarrow$ Brier = 0.64 (Notice how failing here is penalised a lot)

**$\Delta$ AUC with new feature vs no features**

6000 random customers drawn with replacement (sometimes drawn twice)

Recompute AUC, then calculate highest and lowest $\Delta$

Repeat 500 times for a full Bootstrap

In [ ]:
# For completeness sake, I'll see if the Bootstrap changes if applied to Brier scores

def bootstrap_delta(y, p_base, p_new, metric, n=500, seed=0):
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        deltas.append(metric(y[idx], p_new[idx]) - metric(y[idx], p_base[idx]))
    low, high = np.percentile(deltas, [2.5, 97.5])
    return low, high


In [ ]:
y_val = validation[TARGET].to_numpy()
models = {"WoE scorecard": fit_predict_scorecard, "LightGBM": fit_predict_lightgbm}

rows = []
for model_name, fit_predict in models.items():
    predictions = {name: fit_predict(cols) for name, cols in feature_sets.items()}
    base, new = predictions["raw only"], predictions["raw + ratios"]
    auc_ci = bootstrap_delta(y_val, base, new, roc_auc_score)
    brier_ci = bootstrap_delta(y_val, base, new, brier_score_loss)
    for name, p in predictions.items():
        auc = roc_auc_score(y_val, p)
        is_new = name != "raw only"
        rows.append(
            {
                "model": model_name,
                "features": name,
                "AUC": auc,
                "Gini": 2 * auc - 1,
                "Brier": brier_score_loss(y_val, p),
                "ΔAUC 95% CI": f"[{auc_ci[0]:+.4f}, {auc_ci[1]:+.4f}]" if is_new else "",
                "ΔBrier 95% CI": f"[{brier_ci[0]:+.4f}, {brier_ci[1]:+.4f}]" if is_new else "",
            }
        )

results = pd.DataFrame(rows).set_index(["model", "features"])
results.style.format({"AUC": "{:.4f}", "Gini": "{:.4f}", "Brier": "{:.4f}"})


Indeed, the proposed feature fixing changes nothing.